[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/thermodynamics/property_charts/property_charts.ipynb)

# Air Conditioning on the Psychrometric Chart

This example computes the cooling, condensed water, and reheating needed to bring warm humid air to a comfortable supply condition at atmospheric pressure. A cooling coil cools the air until it reaches saturation and then cools it further while excess water vapour condenses into liquid; a reheater then warms the cold saturated air at fixed water content to the supply temperature. Engineers read both stages off the psychrometric chart, which plots water content against temperature alongside curves of constant relative humidity and enthalpy.

A state of moist air is modelled as an antivector in the exterior plane `t0 w0` of temperature `t.dual()` and water content `w.dual()` per kilogram of dry air, with each read back by pairing (`t & state` and `w & state`). Relative humidity and enthalpy are scalar functions over that state plane, and each stage constrains how the state may move through a covector `constraint` whose pairing `constraint & step` vanishes: `w` where no water condenses, and `relative_humidity_slope` along the saturation curve where relative humidity stays at one. Dividing `constraint` by the intrinsic scalar `(t ^ constraint).dual()` gives the change of state per degree warmer under either constraint without coordinate unpacking or matrix solves.

In partial-derivative notation, marching per unit dry-bulb temperature $T$ along saturation $\phi(T, w) = p_v(w)/p_{\text{sat}}(T) = 1$ differentiates $\phi(T, w(T)) = 1$ implicitly to obtain $(dw/dT)_{\phi=1} = -(\partial\phi/\partial T)_w / (\partial\phi/\partial w)_T$, while energy and moisture balances write the coil duty and reheat separately as $q_c = h(T_1, w_1) - h(T_2, w_2) - (w_1 - w_2)\,c_w T_2$ and $q_r = h(T_3, w_2) - h(T_2, w_2)$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
import numpy as np

from numga import Algebra, NumpyContext, concatenate, stack
from examples.util.animation import show_states
from examples.thermodynamics.property_charts import render, scenarios

# Exterior plane of temperature (degrees Celsius) and water content (kg water per kg dry air; grams per kg on the chart).
ga = Algebra("t0w0")
mv = NumpyContext(ga, np.float64).multivector
Scalar = ga.gatype.scalar()                                                   # Scalar+
State = ga.gatype.antivector()                                                # Antivector-
Potential = ga.gatype.vector()                                                # Vector+

# Magnus saturation constants (kPa and degrees Celsius), ratio of molar masses of water and dry air, specific heats
# (kJ / (kg K)), and latent heat of vaporisation at zero degrees Celsius (kJ / kg).
magnus_pressure, magnus_slope, magnus_offset = 0.61078, 17.27, 237.3
molar_ratio = 0.62198
dry_air_heat, vapour_heat, liquid_heat, latent_heat = 1.006, 1.86, 4.186, 2501.0

## The psychrometric chart and constrained directions

Relative humidity compares the vapour's partial pressure with the Magnus saturation pressure, while enthalpy adds the sensible heat of dry air to the latent and sensible heat of the water vapour. Differentiating relative humidity with respect to temperature and water content gives the covector `relative_humidity_slope` that pairs with a step in state to measure how relative humidity changes. At the unsaturated inlet the coil holds water content fixed (`w`), whereas at the saturated coil outlet it holds relative humidity at one (`relative_humidity_slope`), and `constraint / (t ^ constraint).dual()` gives the state step per degree warmer in both cases.

In [ ]:
pressure = scenarios.pressure
inlet_temperature, inlet_relative_humidity = scenarios.inlet_temperature, scenarios.inlet_relative_humidity
coil_temperature, supply_temperature = scenarios.coil_temperature, scenarios.supply_temperature


def saturation_pressure(state: State) -> Scalar:
    """Saturation vapour pressure over liquid water at the state's temperature."""
    warmth = mv.t & state                                                     # [...] Scalar+
    return magnus_pressure * (magnus_slope * warmth / (warmth + magnus_offset)).exp()


def relative_humidity(state: State) -> Scalar:
    """Ratio of the water vapour's partial pressure to the saturation pressure."""
    water = mv.w & state                                                      # [...] Scalar+
    return pressure * water / ((molar_ratio + water) * saturation_pressure(state))


def relative_humidity_slope(state: State) -> Potential:
    """Covector slope pairing with a step in state to give the change in relative humidity."""
    warmth, water = mv.t & state, mv.w & state                                # [...] Scalar+, Scalar+
    saturation = saturation_pressure(state)                                   # [...] Scalar+
    relative = pressure * water / ((molar_ratio + water) * saturation)        # [...] Scalar+
    d_warmth = -relative * magnus_slope * magnus_offset / (warmth + magnus_offset).squared()
    d_water = pressure * molar_ratio / ((molar_ratio + water).squared() * saturation)
    return d_warmth * mv.t + d_water * mv.w                                   # [...] Potential+


def enthalpy(state: State) -> Scalar:
    """Sensible and latent heat of dry air and water vapour per kilogram of dry air."""
    warmth, water = mv.t & state, mv.w & state                                # [...] Scalar+, Scalar+
    return dry_air_heat * warmth + water * (latent_heat + vapour_heat * warmth)


def air_state(warmth: float, relative: float) -> State:
    """Moist air state at a given temperature and relative humidity."""
    dry = mv.t.dual() * warmth + mv.w.dual() * 0                              # [] State-
    vapour = relative * saturation_pressure(dry)                              # [] Scalar+
    return dry + mv.w.dual() * (molar_ratio * vapour / (pressure - vapour))   # [] State-


def constrained_direction(constraint: Potential) -> State:
    """Change of state per degree warmer along `constraint & step == 0`."""
    return (constraint / (mv.t ^ constraint).dual()).retyped(State)           # [...] State-


inlet = air_state(inlet_temperature, inlet_relative_humidity)                 # [] State-
outlet = air_state(coil_temperature, 1.0)                                     # [] State-
outlet_slope = relative_humidity_slope(outlet)                                # [] Potential+
key_states = stack((inlet, outlet))                                           # [2] State-
directions = stack((constrained_direction(mv.w),
                    constrained_direction(outlet_slope)))                     # [2] State-
render.draw_chart(relative_humidity, enthalpy, key_states, directions);

In the chart above, blue curves mark equal relative humidity up to the black saturation curve, shaded grey above where air cannot hold that much vapour, and brown lines mark equal enthalpy with their values printed along saturation. The green arrow at the warm inlet points horizontally at fixed water content, while the green arrow at the cold coil outlet is tangent to the saturation curve.

## Cooling and reheating the air

As the coil cools the air toward `coil_temperature`, water content stays fixed until saturation humidity drops below the inlet water content, after which the excess vapour condenses out onto the coil. The reheater then warms the air from `coil_temperature` to `supply_temperature` at the outlet water content. Subtracting the sensible heat carried away by the liquid condensate at `coil_temperature` from the enthalpy drop across the coil gives the heat removed by the coil.

In [ ]:
def saturation_humidity(state: State) -> Scalar:
    """Maximum water vapour per kilogram of dry air at the state's temperature."""
    vapour = saturation_pressure(state)                                       # [...] Scalar+
    return molar_ratio * vapour / (pressure - vapour)


n_steps, frame_duration_ms = scenarios.n_steps, scenarios.frame_duration_ms
fractions = np.linspace(0, 1, n_steps)
coil_temperatures = inlet_temperature * (1 - fractions) + coil_temperature * fractions
uncondensed = mv.t.dual() * coil_temperatures + mv.w.dual() * (mv.w & inlet)          # [n_steps] State-
condensed = ((mv.w & inlet) - saturation_humidity(uncondensed)).clip(0, np.inf)       # [n_steps] Scalar+
cooled = uncondensed - mv.w.dual() * condensed                                        # [n_steps] State-
reheater_temperatures = coil_temperature * (1 - fractions) + supply_temperature * fractions
reheated = mv.t.dual() * reheater_temperatures + mv.w.dual() * (mv.w & outlet)        # [n_steps] State-
path = concatenate([cooled, reheated[1:]])                                            # [2 * n_steps - 1] State-


def air_states():
    """Yields each air state along the conditioning path."""
    for index in range(path.shape[0]):
        yield path[index]


path_states = show_states(air_states(), render.conditioning, relative_humidity, enthalpy, path, key_states,
                          directions, name="air_conditioning", duration_ms=frame_duration_ms,
                          n_frames=2 * n_steps - 1)

As the air parcel travels down the duct from left to right, its state moves horizontally across the chart until it meets the saturation curve, slides down the curve while liquid water rises in the tray beneath the blue cooling coil, and finally moves horizontally to the right across the orange reheater.

In [ ]:
supply_state = path_states[-1]                                                # [] State-
condensate = (mv.w & inlet) - (mv.w & outlet)                                 # [] Scalar+
cooling = enthalpy(inlet) - enthalpy(outlet) - condensate * liquid_heat * (mv.t & outlet)   # [] Scalar+
reheating = enthalpy(supply_state) - enthalpy(outlet)                         # [] Scalar+
supply_relative_humidity = relative_humidity(supply_state)                    # [] Scalar+

print(f"cooling: {float(cooling.to_array()):.1f} kJ, water removed: {1000 * float(condensate.to_array()):.1f} g, "
      f"reheating: {float(reheating.to_array()):.1f} kJ, per kg dry air")
print(f"supply relative humidity: {float(supply_relative_humidity.to_array()):.0%}")

In [ ]:
# checks
np.testing.assert_allclose(((mv.w & inlet) - (mv.w & supply_state) - condensate).kernel, 0, atol=1e-12)
np.testing.assert_allclose((enthalpy(inlet) - enthalpy(supply_state) - cooling
                            - condensate * liquid_heat * coil_temperature + reheating).kernel, 0, atol=1e-10)
assert np.all(relative_humidity(path) <= 1 + 1e-10)
np.testing.assert_allclose(((mv.t & directions) - 1).kernel, 0, atol=1e-10)
np.testing.assert_allclose((mv.w & directions[0]).kernel, 0, atol=1e-10)
np.testing.assert_allclose((outlet_slope & directions[1]).kernel, 0, atol=1e-10)